In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tabpfn import TabPFNClassifier
from tabpfn_extensions.embedding import TabPFNEmbedding

In [ ]:

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
class StudentModel(nn.Module):
    def __init__(self, in_dim, emb_dim, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            #nn.Linear(256, 256),
            #nn.ReLU(),
            nn.Linear(256, emb_dim),
        )
        self.decoder = nn.Linear(emb_dim, n_classes)

    def forward(self, x):
        z = self.encoder(x)
        logits = self.decoder(z)
        return z, logits

In [ ]:
teacher = TabPFNClassifier(n_estimators=1, random_state=42)
teacher.fit(X_train, y_train)

embedder = TabPFNEmbedding(model=teacher, n_fold=0)

emb_train = embedder.fit_transform(X_train, y_train)
emb_test = embedder.transform(X_test)

if emb_train.ndim == 3:
    emb_train = emb_train[0]
    emb_test = emb_test[0]

teacher_proba_train = teacher.predict_proba(X_train)
teacher_proba_test = teacher.predict_proba(X_test)

x_scaler = StandardScaler().fit(X_train)
emb_scaler = StandardScaler().fit(emb_train)

X_train_s = x_scaler.transform(X_train)
X_test_s = x_scaler.transform(X_test)

emb_train_s = emb_scaler.transform(emb_train)
emb_test_s = emb_scaler.transform(emb_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

emb_train_t = torch.tensor(emb_train_s, dtype=torch.float32)
emb_test_t = torch.tensor(emb_test_s, dtype=torch.float32)

teacher_train_t = torch.tensor(teacher_proba_train, dtype=torch.float32)

n_features = X_train.shape[1]
emb_dim = emb_train.shape[1]
n_classes = teacher_proba_train.shape[1]


student = StudentModel(n_features, emb_dim, n_classes)

opt1 = torch.optim.AdamW(student.encoder.parameters(), lr=1e-3, weight_decay=1e-4)
opt2 = torch.optim.AdamW(student.decoder.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
alpha = 1.0
beta = 1.0

for epoch in range(2000):
    opt1.zero_grad()
    opt2.zero_grad()

    z_hat, logits_hat = student(X_train_t)
    log_probs_hat = F.log_softmax(logits_hat, dim=1)

    loss_emb = F.mse_loss(z_hat, emb_train_t)
    loss_kl = F.kl_div(log_probs_hat, teacher_train_t, reduction="batchmean")
    loss=loss_kl*alpha+loss_emb*beta
    loss.backward()
    opt1.step()
    opt1.zero_grad()
    opt2.step()
    opt2.zero_grad()

    #loss = alpha * loss_emb + beta * loss_kl
    #loss.backward()

    if (epoch + 1) % 200 == 0:
        print(
            f"epoch={epoch+1:4d}  "
            f"loss={loss.item():.6f}  "
            f"emb={loss_emb.item():.6f}  "
            f"kl={loss_kl.item():.6f}"
        )


In [ ]:
with torch.no_grad():
    _, logits_test = student(X_test_t)
    student_proba_test = F.softmax(logits_test, dim=1).cpu().numpy()

teacher_pred = teacher_proba_test.argmax(axis=1)
student_pred = student_proba_test.argmax(axis=1)

teacher_acc = (teacher_pred == y_test).mean()
student_acc = (student_pred == y_test).mean()
agreement = (teacher_pred == student_pred).mean()

kl = np.mean(
    np.sum(
        teacher_proba_test
        * (
            np.log(teacher_proba_test + 1e-9)
            - np.log(student_proba_test + 1e-9)
        ),
        axis=1,
    )
)

In [ ]:

print("\nRESULTS")
print(f"Teacher acc       : {teacher_acc:.4f}")
print(f"Student acc       : {student_acc:.4f}")
print(f"Teacher agreement : {agreement:.4f}")
print(f"KL(T || S)        : {kl:.6f}")

## Evaluate every classification dataset

The single-dataset run above (breast cancer) is generalized below into a sweep over **every**
classification dataset in the benchmark suite (`benchmark_datasets.OpenMLBenchmark`). For each
dataset and split it fits the TabPFN teacher, distills the `StudentModel` (encoder regressed onto
the TabPFN embedding with MSE + decoder matched to the teacher probabilities with KL), and also
trains a **from-scratch baseline**: the *same architecture* trained directly on the hard labels with
cross-entropy (no teacher, no embedding). The baseline shows how much the distillation actually adds
over plain supervised learning at equal capacity.

Recorded per dataset:

- `teacher_acc` / `student_acc` / `baseline_acc` — test accuracy of each
- `agreement` — fraction of test rows where student and teacher predict the same class
- `teacher_auc` / `student_auc` / `baseline_auc` — ROC AUC (one-vs-rest macro for multiclass)
- `kl` — mean `KL(teacher || student)` on the test set

Everything is unified over the number of classes (softmax + KL throughout), so the same code handles
binary and multiclass datasets. Results are aggregated into a per-dataset table and an overall summary.


In [23]:
import pandas as pd
from benchmark_datasets import OpenMLBenchmark, _roc_auc
from aug_dataset import aug_dataset
benchmarks = OpenMLBenchmark("classification")
STUDENT_METRICS = [
    "teacher_acc", "student_acc", "baseline_acc", "agreement",
    "teacher_auc", "student_auc", "baseline_auc", "kl",
]


def _student_proba(student, x_scaler, X):
    """Class probabilities from a trained StudentModel (uses X only -- no teacher at inference)."""
    with torch.no_grad():
        _, logits = student(torch.tensor(x_scaler.transform(X), dtype=torch.float32))
        return F.softmax(logits, dim=1).cpu().numpy()


def _fit_distilled_student(X_train, y_train, n_classes, *, epochs, alpha, beta, lr, weight_decay):
    """Fit TabPFN + distill the StudentModel (encoder->embedding MSE, decoder->teacher KL).

    Returns ``(teacher, student, x_scaler, emb_dim)``.
    """
    teacher = TabPFNClassifier(n_estimators=1, random_state=42).fit(X_train, y_train)
    embedder = TabPFNEmbedding(model=teacher, n_fold=0)
    emb_train = embedder.fit_transform(X_train, y_train)
    if emb_train.ndim == 3:
        emb_train = emb_train[0]

    x_scaler = StandardScaler().fit(X_train)
    emb_scaler = StandardScaler().fit(emb_train)
    X_train_t = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32)
    emb_train_t = torch.tensor(emb_scaler.transform(emb_train), dtype=torch.float32)
    teacher_train_t = torch.tensor(teacher.predict_proba(X_train), dtype=torch.float32)

    student = StudentModel(X_train.shape[1], emb_train.shape[1], n_classes)
    #opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    opt1 = torch.optim.AdamW(student.encoder.parameters(), lr=1e-3, weight_decay=1e-4)
    opt2 = torch.optim.AdamW(student.decoder.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        opt1.zero_grad()
        opt2.zero_grad()
        z_hat, logits_hat = student(X_train_t)
        log_probs_hat = F.log_softmax(logits_hat, dim=1)
        #loss = (alpha * F.mse_loss(z_hat, emb_train_t)
        #        + beta * F.kl_div(F.log_softmax(logits_hat, dim=1), teacher_train_t, reduction="batchmean"))
        loss_emb = F.mse_loss(z_hat, emb_train_t)
        loss_kl = F.kl_div(log_probs_hat, teacher_train_t, reduction="batchmean")
        loss = loss_emb*alpha+loss_kl*beta
        loss.backward()
        opt1.step()
        opt1.zero_grad()
        opt2.step()
        opt2.zero_grad()
        #loss.backward()
        #opt.step()
    return teacher, student, x_scaler, emb_train.shape[1]


def _fit_scratch_student(X_train, y_train, n_classes, emb_dim, *, epochs, lr, weight_decay):
    """Baseline: the *same architecture* as the distilled student, but trained directly on the
    hard labels with cross-entropy (no teacher, no embedding). Isolates how much the distillation
    actually adds over plain supervised learning at equal capacity.
    """
    x_scaler = StandardScaler().fit(X_train)
    X_train_t = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)

    student = StudentModel(X_train.shape[1], emb_dim, n_classes)
    opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad()
        _, logits = student(X_train_t)
        loss = ce(logits, y_train_t)
        loss.backward()
        opt.step()
    return student, x_scaler


def evaluate_student_on_dataset(
    ds, *, n_splits=5, train_size=32, epochs=1000, alpha=1.0, beta=1.0, lr=1e-3, weight_decay=1e-4
):
    """Distill the student against TabPFN across repeated splits of one ``LoadedDataset``, and
    compare against a same-architecture from-scratch baseline. Returns ``{metric: [...]}`` with
    one entry per split.
    """
    X, y = ds.X, ds.y
    classes = np.unique(y)
    n_classes = len(classes)
    out = {k: [] for k in STUDENT_METRICS}

    for train_idx, test_idx in benchmarks.splits(
        X, y, task="classification", n_splits=n_splits, train_size=train_size
    ):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        teacher, student, x_scaler, emb_dim = _fit_distilled_student(
            X_train, y_train, n_classes,
            epochs=epochs, alpha=alpha, beta=beta, lr=lr, weight_decay=weight_decay,
        )
        baseline, base_scaler = _fit_scratch_student(
            X_train, y_train, n_classes, emb_dim, epochs=epochs, lr=lr, weight_decay=weight_decay,
        )

        teacher_proba_test = teacher.predict_proba(X_test)
        student_proba_test = _student_proba(student, x_scaler, X_test)
        baseline_proba_test = _student_proba(baseline, base_scaler, X_test)

        teacher_pred = teacher_proba_test.argmax(axis=1)
        student_pred = student_proba_test.argmax(axis=1)
        baseline_pred = baseline_proba_test.argmax(axis=1)

        out["teacher_acc"].append((teacher_pred == y_test).mean())
        out["student_acc"].append((student_pred == y_test).mean())
        out["baseline_acc"].append((baseline_pred == y_test).mean())
        out["agreement"].append((teacher_pred == student_pred).mean())
        out["teacher_auc"].append(_roc_auc(y_test, teacher_proba_test, classes))
        out["student_auc"].append(_roc_auc(y_test, student_proba_test, classes))
        out["baseline_auc"].append(_roc_auc(y_test, baseline_proba_test, classes))
        out["kl"].append(float(np.mean(np.sum(
            teacher_proba_test
            * (np.log(teacher_proba_test + 1e-9) - np.log(student_proba_test + 1e-9)),
            axis=1,
        ))))

    return out


def evaluate_all_classification_students(datasets=None, *, verbose=True, **kwargs):
    """Distill the student over every classification dataset, with a from-scratch baseline, and
    aggregate. Returns ``(per_dataset, summary)`` DataFrames. ``**kwargs`` are forwarded to
    ``evaluate_student_on_dataset`` (``n_splits``, ``train_size``, ``epochs``, ``alpha``, ``beta``,
    ``lr``, ``weight_decay``).
    """
    specs = benchmarks.specs
    if datasets is not None:
        wanted = set(datasets)
        specs = [s for s in specs if s.name in wanted]

    rows = []
    for spec in specs:
        ds = benchmarks.load(spec)
        try:
            res = evaluate_student_on_dataset(ds, **kwargs)
        except Exception as exc:                       # keep the sweep going on a bad dataset
            print(f"[skip] {spec.name}: {exc}")
            continue
        row = {"dataset": spec.name, "n_rows": spec.n_rows, "n_classes": ds.n_classes}
        for k in STUDENT_METRICS:
            row[k] = float(np.nanmean(res[k]))
        rows.append(row)
        if verbose:
            print(
                f"{spec.name:24s} (k={ds.n_classes}) "
                f"teacher={row['teacher_acc']:.3f} student={row['student_acc']:.3f} "
                f"baseline={row['baseline_acc']:.3f} agree={row['agreement']:.3f} kl={row['kl']:.3f}"
            )

    per_dataset = pd.DataFrame(rows)
    summary = pd.DataFrame(
        {
            "metric": STUDENT_METRICS,
            "mean": [per_dataset[k].mean() for k in STUDENT_METRICS],
            "std": [per_dataset[k].std() for k in STUDENT_METRICS],
        }
    )
    return per_dataset, summary


In [25]:
# Run the student distillation (+ from-scratch baseline) over every classification dataset.
# Heavy: ~50 datasets x n_splits x (TabPFN fit + embedding + epochs of the 256-wide student x2).
# Pass e.g. datasets=["sonar", "wine"] or a smaller epochs/n_splits for a quick smoke test.
per_dataset, summary = evaluate_all_classification_students()

print("\n=== Aggregate across all classification datasets ===")
display(summary.round(4))
display(per_dataset.round(4))

# Mean ROC AUC across all datasets for the teacher, student, and baseline.
auc_means = summary.set_index("metric")["mean"]
print("\n=== Mean ROC AUC across all classification datasets ===")
print(f"teacher  : {auc_means['teacher_auc']:.4f}")
print(f"student  : {auc_means['student_auc']:.4f}")
print(f"baseline : {auc_means['baseline_auc']:.4f}")


tic-tac-toe              (k=2) teacher=0.673 student=0.651 baseline=0.669 agree=0.768 kl=0.644
monks-problems-2         (k=2) teacher=0.655 student=0.588 baseline=0.591 agree=0.812 kl=0.216
sonar                    (k=2) teacher=0.713 student=0.723 baseline=0.755 agree=0.826 kl=1.023
ionosphere               (k=2) teacher=0.853 student=0.846 baseline=0.818 agree=0.881 kl=0.872
vehicle                  (k=4) teacher=0.643 student=0.643 baseline=0.608 agree=0.731 kl=1.833
wdbc                     (k=2) teacher=0.924 student=0.906 baseline=0.937 agree=0.945 kl=0.213
diabetes                 (k=2) teacher=0.685 student=0.654 baseline=0.696 agree=0.710 kl=1.409
ilpd                     (k=2) teacher=0.687 student=0.564 baseline=0.642 agree=0.634 kl=2.816
balance-scale            (k=3) teacher=0.821 student=0.819 baseline=0.862 agree=0.855 kl=0.291
blood-transfusion        (k=2) teacher=0.741 student=0.687 baseline=0.687 agree=0.826 kl=0.196
titanic                  (k=2) teacher=0.735 stude

,metric,mean,std
0,teacher_acc,0.7729,0.1653
1,student_acc,0.7240,0.1818
2,baseline_acc,0.7476,0.1775
3,agreement,0.8012,0.1549
4,teacher_auc,0.7811,0.1591
5,student_auc,0.7481,0.1641
6,baseline_auc,0.7564,0.1618
7,kl,0.9675,0.9635


,dataset,n_rows,n_classes,teacher_acc,student_acc,baseline_acc,agreement,teacher_auc,student_auc,baseline_auc,kl
0,tic-tac-toe,958,2,0.6730,0.6512,0.6693,0.7678,0.6817,0.6668,0.6675,0.6444
1,monks-problems-2,601,2,0.6548,0.5877,0.5909,0.8120,0.5894,0.5497,0.5661,0.2155
2,sonar,208,2,0.7125,0.7227,0.7545,0.8261,0.7904,0.8057,0.8246,1.0226
3,ionosphere,351,2,0.8533,0.8458,0.8182,0.8809,0.9395,0.8644,0.8529,0.8716
4,vehicle,846,4,0.6428,0.6425,0.6079,0.7312,0.8606,0.8438,0.8050,1.8328
5,wdbc,569,2,0.9244,0.9061,0.9367,0.9453,0.9780,0.9575,0.9772,0.2133
6,diabetes,768,2,0.6851,0.6541,0.6959,0.7098,0.7932,0.6713,0.7314,1.4085
7,ilpd,583,2,0.6871,0.5641,0.6417,0.6338,0.7019,0.6032,0.6353,2.8160
8,balance-scale,625,3,0.8209,0.8189,0.8617,0.8546,0.8654,0.8812,0.9079,0.2915
9,blood-transfusion,748,2,0.7413,0.6866,0.6869,0.8263,0.6737,0.6356,0.6340,0.1959



=== Mean ROC AUC across all classification datasets ===
teacher  : 0.7811
student  : 0.7481
baseline : 0.7564


In [26]:
summary.to_csv("results/student_model_32_sum.csv")
per_dataset.to_csv("results/student_model_32_dataset.csv")